In [233]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [234]:
# Define the path to your synthetic data relative to the notebook
data_path = Path("..") / "data" / "synthetic"

# Load the CSVs
users = pd.read_csv(data_path / "users.csv")
products = pd.read_csv(data_path / "products.csv")
vendors = pd.read_csv(data_path / "vendors.csv")
purchases = pd.read_csv(data_path / "purchases.csv")

In [235]:


import numpy as np

# Set target size
target_user_count = 500
target_purchase_count = int(1000 * (target_user_count / users.shape[0]))

# Generate new user entries
new_users = users.sample(n=target_user_count - users.shape[0], replace=True).copy()
new_users = new_users.reset_index(drop=True)
new_users['user_id'] = range(users['user_id'].max() + 1, users['user_id'].max() + 1 + new_users.shape[0])

# Concatenate with original users
users_extended = pd.concat([users, new_users], ignore_index=True)

# Generate new purchases
new_purchases = []
for _ in range(target_purchase_count - purchases.shape[0]):
    user_id = np.random.choice(users_extended['user_id'])
    product_id = np.random.choice(products['product_id'])
    vendor_id = np.random.choice(vendors['vendor_id'])
    amount_spent = round(np.random.uniform(5.0, 100.0), 2)
    new_purchases.append([user_id, product_id, vendor_id, amount_spent])

new_purchases_df = pd.DataFrame(new_purchases, columns=['user_id', 'product_id', 'vendor_id', 'amount_spent'])

# Generate purchase_id for new records
new_purchases_df['purchase_id'] = range(purchases['purchase_id'].max() + 1, purchases['purchase_id'].max() + 1 + new_purchases_df.shape[0])

# Reorder columns
new_purchases_df = new_purchases_df[['user_id', 'product_id', 'vendor_id', 'amount_spent']]

# Combine with original purchases
purchases_extended = pd.concat([purchases, new_purchases_df], ignore_index=True)

# Show resulting shapes
users_extended.shape, purchases_extended.shape

((500, 5), (2500, 5))

In [236]:
full["vendor_type"].value_counts()

Franchise      881
Independent    847
Online         772
Name: vendor_type, dtype: int64

In [238]:
# Join all tables

full = users_extended \
    .merge(purchases_extended, on="user_id", how="outer") \
    .merge(products, on="product_id") \
    .merge(vendors, on="vendor_id")

In [239]:
full

,user_id,age,region,account_type,preferred_device,purchase_id,product_id,vendor_id,amount_spent,category,price,location,vendor_type
0,1,56,South,Enterprise,Tablet,760.0,34.0,13.0,226.69,Books,400.71,Rural,Independent
1,119,59,West,Enterprise,Mobile,727.0,34.0,13.0,540.91,Books,400.71,Rural,Independent
2,89,38,East,Premium,Desktop,NaN,48.0,13.0,79.33,Books,187.61,Rural,Independent
3,1,56,South,Enterprise,Tablet,914.0,95.0,13.0,61.73,Electronics,44.72,Rural,Independent
4,184,45,North,Free,Tablet,NaN,90.0,13.0,19.60,Electronics,84.54,Rural,Independent
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2495,229,61,East,Free,Tablet,NaN,14.0,3.0,20.61,Electronics,322.23,Suburban,Independent
2496,424,53,South,Free,Mobile,NaN,14.0,3.0,47.51,Electronics,322.23,Suburban,Independent
2497,39,61,West,Enterprise,Desktop,NaN,35.0,3.0,12.39,Electronics,315.81,Suburban,Independent
2498,196,69,South,Free,Mobile,NaN,35.0,3.0,21.04,Electronics,315.81,Suburban,Independent


In [240]:
def compute_user_labels(purchases, products, vendors, min_matches=2, min_price=10):
    """
    Returns a DataFrame with user_id and label.
    A user is labeled 1 if they made at least `min_matches` purchases from Online vendors
    in the Electronics or Toys category, with price > `min_price`.
    """
    # Merge to unified view
    full = purchases \
        .merge(products[['product_id', 'category', 'price']], on='product_id') \
        .merge(vendors[['vendor_id', 'vendor_type']], on='vendor_id')
    
    # Define the logical condition
    criteria = (
        (full['vendor_type'].isin(["Independent", "Online"])) &
        (full['category'].isin(['Electronics', 'Toys', 'Books', 'Clothing'])) &
        (full['price'] > min_price)
    )
    
    # Count qualifying purchases per user
    user_agg = full[criteria].groupby('user_id').size().reset_index(name='match_count')
    user_agg['label'] = (user_agg['match_count'] >= min_matches).astype(int)

    # Merge back to full user list to assign 0 to unmatched users
    all_users = purchases['user_id'].unique()
    user_labels = pd.DataFrame({'user_id': all_users})
    user_labels = user_labels.merge(user_agg[['user_id', 'label']], on='user_id', how='left')
    user_labels['label'] = user_labels['label'].fillna(0).astype(int)
    
    return user_labels

In [241]:
def add_label_noise(user_labels, noise_rate=0.01, seed=42):
    """
    Randomly flips `noise_rate` proportion of labels (symmetric label noise).
    """
    np.random.seed(seed)
    user_labels = user_labels.copy()
    flip_mask = np.random.rand(len(user_labels)) < noise_rate
    user_labels.loc[flip_mask, 'label'] = 1 - user_labels.loc[flip_mask, 'label']
    return user_labels

In [242]:
# Compute labels (based on actual behavior in merged tables)
user_labels = compute_user_labels(purchases_extended, products, vendors)

# Optionally inject noise (e.g., 10%)
user_labels_noisy = add_label_noise(user_labels, noise_rate=0.1)

In [243]:
user_labels.sum()

user_id    117250
label         337
dtype: int64

In [244]:
user_labels_noisy.sum()

user_id    117250
label         332
dtype: int64

In [245]:
# Save the CSVs
#users_extended.to_csv(data_path / "users.csv")
#purchases_extended.to_csv(data_path / "purchases.csv")

In [246]:
task_table = users_extended.merge(user_labels, on="user_id", how="left")[["user_id", "label"]]

In [247]:
task_table.fillna(0, inplace=True)

In [248]:
task_table.to_csv(os.path.join(data_path, "task.csv"), index=False)

In [249]:
task = pd.read_csv(data_path / "task.csv")

In [250]:
label_dict = dict(zip(task['user_id'], task['label']))

In [251]:
task.sum()

user_id    125250.0
label         337.0
dtype: float64